# Brown Weather Framework

A basic DATA1030 framework with three interfaces: Data, Models, and Analysis.
Running all cells only imports the pipeline and previews the plan. Future actions are commented out.

## Setup

Use the `weather1030` environment. Launch from the project root or its `src` folder.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src" / "weather_project").is_dir():
    ROOT = cwd
elif (cwd / "weather_project").is_dir():
    ROOT = cwd.parent
else:
    raise RuntimeError("Open this notebook from the project root or its src folder.")

source_dir = str(ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from weather_project.pipeline import run_pipeline


## Data

**Location and source:** Brown University (41.82682, −71.40293), Open-Meteo ERA5, 2001–2025. Collection requests 15 daily variables using `America/New_York`, °C, km/h, and mm.

For target date **d**, samples flatten individual observations from **28 recent days (d−28 … d−1)** and **35 historical positions**. Historical anchors use the same calendar date in each of the five preceding years, with **±3 calendar days** around each anchor. This is **four recent weeks plus five separate historical weeks**, not nine continuous weeks.

Every position supplies **15 raw API variables plus two month/day sine/cosine encodings**, using reference year 2000 (366 days). Two further encodings describe the target date. Thus **63 × 17 + 2 = 1,073 raw X columns**, excluding metadata and y. Historical values are kept individually; no seasonal means or standard deviations are computed.

February 29 anchors use February 28 in non-leap years. Historical windows may cross year boundaries and always precede the target. Every required calendar date must be present; the first five years plus three initial boundary days are warm-up.

The **only y** is `target_mean_temperature`. Target-date boundaries are training through **2017-12-31**, validation through **2021-12-31**, and testing afterward. This is a rolling one-day retrospective ERA5 experiment: earlier observed validation/test days may become later inputs; fitting and tuning remain within their assigned periods.

The 9,131-day collection and API response are bundled in `data/`. The next cell previews the plan. To rebuild files, uncomment and run `collect` first (reusing the cache), then `prepare`. No model fitting is performed.

Field definitions and units: [data dictionary](../data/README.md). Collection and feature logic: [data.py](weather_project/data.py), [features.py](weather_project/features.py).


In [ ]:
plan = run_pipeline(stage="plan", project_root=ROOT)
plan


In [ ]:
# Step 1: fetch and save the raw ERA5 response, daily.csv, metadata, and quality checks.
# collection = run_pipeline(stage="collect", project_root=ROOT)

# Step 2: build samples.csv, feature_columns.json, and data_dictionary.csv.
# preparation = run_pipeline(stage="prepare", project_root=ROOT)

# Optional: load prepared daily data for explicit analysis.
# import pandas as pd
# daily = pd.read_csv(ROOT / "data" / "processed" / "daily.csv", parse_dates=["date"])
# samples = pd.read_csv(ROOT / "data" / "processed" / "samples.csv")


## Models

`build_models(task="regression", seed=1030)` returns unfitted Ridge and Random Forest pipelines. Select the **1,073 raw X columns** with `feature_columns.json`; identifiers, split labels, and y are excluded.

WMO `weather_code` is categorical and is one-hot encoded by preprocessing. Wind direction becomes sine/cosine. These transformations can change the feature count. Nothing fits, tunes, or evaluates a model automatically.


In [ ]:
# import json
# from weather_project.modeling import build_models

# feature_columns = json.loads((ROOT / "data/processed/feature_columns.json").read_text())
# train = samples["split"].eq("train")
# X_train = samples.loc[train, feature_columns]
# y_train = samples.loc[train, "target_mean_temperature"]
# regression_models = build_models(task="regression", seed=1030)
# Future preprocessing must fit on training data only; no fitting is performed here.


## Analysis

The midterm EDA is complete: [report PDF](../report/Report.pdf), [eight original figures](../figures/README.md). The figures use Bill's separate 6,941-day, 19-field exploratory collection; the modeling collection has 9,131 days and 15 weather fields.

The helper calls below remain optional. `data_overview` summarizes supplied daily data; `plot_temperature` plots it. `plot_scores` requires a score table from a future experiment.

Models have not been fitted. Future feature comparisons and tuning must respect the chronological splits. The exploratory figures cover the full period, including the validation and test years.


In [ ]:
# from weather_project.analysis import data_overview, plot_temperature, plot_scores

# overview = data_overview(daily)
# plot_temperature(daily)

# After explicitly running and saving an experiment elsewhere:
# scores = pd.read_csv(ROOT / "results" / "scores.csv")
# plot_scores(scores, metric="mae")


Weather data: [Open-Meteo / ERA5](https://open-meteo.com/en/docs/historical-weather-api), CC BY 4.0; code: MIT.
